In [ ]:
search_text = "Nostalgic"

import json
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import boto3

# Load the icon embeddings
with open("icon-index-embeddings.json", "r") as f:
    icon_embeddings = json.load(f)

# Initialize Bedrock client
bedrock_client = boto3.client("bedrock-runtime")


# Generate embedding for search text using Nova Multimodal Embeddings
def get_text_embedding(text):
    request_body = {
        "taskType": "SINGLE_EMBEDDING",
        "singleEmbeddingParams": {
            "embeddingPurpose": "GENERIC_RETRIEVAL",
            "embeddingDimension": 3072,
            "text": {"truncationMode": "END", "value": text},
        },
    }

    response = bedrock_client.invoke_model(
        modelId="amazon.nova-2-multimodal-embeddings-v1:0",
        body=json.dumps(request_body),
        contentType="application/json",
        accept="application/json",
    )

    response_body = json.loads(response["body"].read())
    return response_body["embeddings"][0]["embedding"]


# Get embedding for search text
search_embedding = get_text_embedding(search_text)

# Calculate cosine similarities
similarities = []
icon_names = []

for icon in icon_embeddings:
    icon_name = icon["iconId"]
    icon_embedding = icon["embedding"]
    similarity = cosine_similarity([search_embedding], [icon_embedding])[0][0]
    similarities.append(similarity)
    icon_names.append(icon_name)

# Find the top 5 matching icons
top_indices = np.argsort(similarities)[-5:][::-1]  # Get top 5 indices in descending order

print("Top 5 matching icons:")
print("-" * 40)
for i, idx in enumerate(top_indices, 1):
    icon_name = icon_names[idx]
    similarity_score = similarities[idx]
    print(f"{i}. {icon_name}: {similarity_score:.4f}")

Closest matching icon: SkipBack
Similarity score: 0.1391
